In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 6.3 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

Neo4j Python Driver Version: 6.3.1


1.เริ่มติดต่อใช้งาน Neo4j online server

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://0d30247f.databases.neo4j.io"
USERNAME = "0d30247f"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

Password: ··········
Connected successfully


หา home database

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

for record in result.records:
    print(record.data())

{'name': '0d30247f', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-2e58c6e6c448-1-0176.production-orch-0703.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


เก็บชื่อ database

In [ ]:
DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

Database = 0d30247f


หลังจากนั้น Query ทุกคำสั่งใช้:

In [ ]:
result = driver.execute_query(
    """
    RETURN 'Hello Neo4j' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])

Hello Neo4j


ทดสอบใช้งานฐานข้อมูลจริง

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])

จำนวน Node = 1


สร้าง Node

In [ ]:
result = driver.execute_query(
    """
    CREATE (u:User {
        name: 'Alice'
    })

    RETURN u
    """,
    database_=DATABASE
)

print(result.records[0]["u"])

<Node element_id='4:26e79ae0-c513-441f-907f-5498c53b2777:0' labels=frozenset({'User'}) properties={'name': 'Alice'}>


ตรวจสอบผลลัพธ์

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)

    RETURN
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'name': 'Alice'}


ทดลอง where

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)

    WHERE u.name = 'Alice'

    RETURN
        u.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'name': 'Alice'}


6.สร้าง constrain

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique
    IF NOT EXISTS

    FOR (u:User)

    REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

print("User constraint created")

User constraint created


In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT car_name_unique
    IF NOT EXISTS

    FOR (c:Car)

    REQUIRE c.name IS UNIQUE
    """,
    database_=DATABASE
)

print("Car constraint created")

Car constraint created


เพิ่ม student หลายๆคน

In [ ]:
users = [
    {"name": "Alice"},
    {"name": "Bob"},
    {"name": "Charlie"},
    {"name": "David"},
    {"name": "Emma"},
    {"name": "Frank"},
    {"name": "Grace"},
    {"name": "Henry"},
    {"name": "Ivy"},
    {"name": "Jack"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $users AS row

    MERGE (u:User {
        name: row.name
    })

    RETURN count(u) AS total
    """,

    users=users,
    database_=DATABASE
)

print(
    "จำนวน User ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน User ที่ประมวลผล = 10


In [ ]:
cars = [
    {"name": "Toyota Yaris",   "brand": "Toyota"},
    {"name": "Honda City",     "brand": "Honda"},
    {"name": "Toyota Corolla", "brand": "Toyota"},
    {"name": "Honda Civic",    "brand": "Honda"},
    {"name": "Mazda 3",        "brand": "Mazda"},
    {"name": "Nissan Almera",  "brand": "Nissan"},
    {"name": "Ford Ranger",    "brand": "Ford"},
    {"name": "Toyota Camry",   "brand": "Toyota"},
    {"name": "Honda Accord",   "brand": "Honda"},
    {"name": "Mazda CX-5",     "brand": "Mazda"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $cars AS row

    MERGE (c:Car {
        name: row.name
    })

    SET c.brand = row.brand

    RETURN count(c) AS total
    """,

    cars=cars,
    database_=DATABASE
)

print(
    "จำนวน Car ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน Car ที่ประมวลผล = 10


ดู student และ book ทั้งหมด

In [ ]:
result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:User OR n:Car

    RETURN
        labels(n) AS type,
        properties(n) AS data
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'type': ['User'], 'data': {'name': 'Alice'}}
{'type': ['User'], 'data': {'name': 'Bob'}}
{'type': ['User'], 'data': {'name': 'Charlie'}}
{'type': ['User'], 'data': {'name': 'David'}}
{'type': ['User'], 'data': {'name': 'Emma'}}
{'type': ['User'], 'data': {'name': 'Frank'}}
{'type': ['User'], 'data': {'name': 'Grace'}}
{'type': ['User'], 'data': {'name': 'Henry'}}
{'type': ['User'], 'data': {'name': 'Ivy'}}
{'type': ['User'], 'data': {'name': 'Jack'}}
{'type': ['Car'], 'data': {'name': 'Toyota Yaris', 'brand': 'Toyota'}}
{'type': ['Car'], 'data': {'name': 'Honda City', 'brand': 'Honda'}}
{'type': ['Car'], 'data': {'name': 'Toyota Corolla', 'brand': 'Toyota'}}
{'type': ['Car'], 'data': {'name': 'Honda Civic', 'brand': 'Honda'}}
{'type': ['Car'], 'data': {'name': 'Mazda 3', 'brand': 'Mazda'}}
{'type': ['Car'], 'data': {'name': 'Nissan Almera', 'brand': 'Nissan'}}
{'type': ['Car'], 'data': {'name': 'Ford Ranger', 'brand': 'Ford'}}
{'type': ['Car'], 'data': {'name': 'Toyota Camry', 'brand'

Step 10 — สร้าง Relationship FRIEND_OF

In [ ]:
likes = [
    # Alice
    {"user": "Alice", "car": "Toyota Yaris"},
    {"user": "Alice", "car": "Honda City"},
    {"user": "Alice", "car": "Toyota Corolla"},

    # Bob
    {"user": "Bob", "car": "Toyota Yaris"},
    {"user": "Bob", "car": "Honda City"},
    {"user": "Bob", "car": "Toyota Corolla"},
    {"user": "Bob", "car": "Honda Civic"},

    # Charlie
    {"user": "Charlie", "car": "Toyota Yaris"},
    {"user": "Charlie", "car": "Toyota Corolla"},
    {"user": "Charlie", "car": "Mazda 3"},

    # David
    {"user": "David", "car": "Toyota Corolla"},
    {"user": "David", "car": "Honda Civic"},
    {"user": "David", "car": "Mazda 3"},
    {"user": "David", "car": "Nissan Almera"},

    # Emma
    {"user": "Emma", "car": "Honda Civic"},
    {"user": "Emma", "car": "Mazda 3"},
    {"user": "Emma", "car": "Nissan Almera"},

    # Frank
    {"user": "Frank", "car": "Nissan Almera"},
    {"user": "Frank", "car": "Ford Ranger"},
    {"user": "Frank", "car": "Toyota Camry"},

    # Grace
    {"user": "Grace", "car": "Ford Ranger"},
    {"user": "Grace", "car": "Toyota Camry"},
    {"user": "Grace", "car": "Honda Accord"},

    # Henry
    {"user": "Henry", "car": "Toyota Camry"},
    {"user": "Henry", "car": "Honda Accord"},
    {"user": "Henry", "car": "Mazda CX-5"},

    # Ivy
    {"user": "Ivy", "car": "Honda Accord"},
    {"user": "Ivy", "car": "Mazda CX-5"},
    {"user": "Ivy", "car": "Toyota Yaris"},

    # Jack
    {"user": "Jack", "car": "Mazda CX-5"},
    {"user": "Jack", "car": "Toyota Yaris"},
    {"user": "Jack", "car": "Honda City"}
]

print("จำนวน LIKES =", len(likes))

จำนวน LIKES = 32


In [ ]:
driver.execute_query(
    """
    UNWIND $likes AS row

    MATCH
        (u:User {name: row.user}),
        (c:Car {name: row.car})

    MERGE
        (u)-[:LIKES]->(c)
    """,

    likes=likes,
    database_=DATABASE
)

print("สร้าง LIKES สำเร็จ")

สร้าง LIKES สำเร็จ


In [ ]:
result = driver.execute_query(
    """
    MATCH (:User)-[r:LIKES]->(:Car)
    RETURN count(r) AS total
    """,
    database_=DATABASE
)

print("จำนวน LIKES =", result.records[0]["total"])

จำนวน LIKES = 32


Step 11 — Query ความสัมพันธ์ Alice ชอบรถรุ่นอะไรบ้าง

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User {name:'Alice'})
        -[:LIKES]->
        (car:Car)

    RETURN car.name AS car
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["car"])

Toyota Yaris
Honda City
Toyota Corolla


Step 12 — สร้าง TEST_DROVE Relationship

In [ ]:
test_drives = [
    {"user": "Alice",   "car": "Toyota Yaris",   "date": "2026-09-01"},
    {"user": "Bob",     "car": "Honda Civic",    "date": "2026-09-02"},
    {"user": "Charlie", "car": "Mazda 3",        "date": "2026-09-03"},
    {"user": "David",   "car": "Nissan Almera",  "date": "2026-09-04"},
    {"user": "Alice",   "car": "Honda Civic",    "date": "2026-09-05"},
    {"user": "Emma",    "car": "Mazda 3",        "date": "2026-09-06"},
    {"user": "Frank",   "car": "Ford Ranger",    "date": "2026-09-07"},
    {"user": "Grace",   "car": "Toyota Camry",   "date": "2026-09-08"},
    {"user": "Henry",   "car": "Mazda CX-5",     "date": "2026-09-09"},
    {"user": "Jack",    "car": "Toyota Yaris",   "date": "2026-09-10"}
]

สร้าง Relationship

In [ ]:
driver.execute_query(
    """
    UNWIND $test_drives AS row

    MATCH
        (u:User {name: row.user}),
        (c:Car {name: row.car})

    MERGE
        (u)-[r:TEST_DROVE]->(c)

    SET
        r.test_date = date(row.date)
    """,

    test_drives=test_drives,
    database_=DATABASE
)

print("สร้าง TEST_DROVE สำเร็จ")

สร้าง TEST_DROVE สำเร็จ


Step 13 — ดูว่าใครทดลองขับรถอะไร

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[r:TEST_DROVE]->
        (c:Car)

    RETURN
        u.name AS user,
        c.name AS car,
        r.test_date AS date

    ORDER BY user, date
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["user"],
        "->",
        record["car"],
        "->",
        record["date"]
    )

Alice -> Toyota Yaris -> 2026-09-01
Alice -> Honda Civic -> 2026-09-05
Bob -> Honda Civic -> 2026-09-02
Charlie -> Mazda 3 -> 2026-09-03
David -> Nissan Almera -> 2026-09-04
Emma -> Mazda 3 -> 2026-09-06
Frank -> Ford Ranger -> 2026-09-07
Grace -> Toyota Camry -> 2026-09-08
Henry -> Mazda CX-5 -> 2026-09-09
Jack -> Toyota Yaris -> 2026-09-10


Step 14 — นำผลลัพธ์มาแสดงเป็นตาราง Pandas

In [ ]:
import pandas as pd

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[r:TEST_DROVE]->
        (c:Car)

    RETURN
        u.name AS user,
        c.name AS car,
        c.brand AS brand,
        r.test_date AS test_date

    ORDER BY test_date
    """,
    database_=DATABASE
)

data = [
    record.data()
    for record in result.records
]

df = pd.DataFrame(data)

df

,user,car,brand,test_date
0,Alice,Toyota Yaris,Toyota,2026-09-01
1,Bob,Honda Civic,Honda,2026-09-02
2,Charlie,Mazda 3,Mazda,2026-09-03
3,David,Nissan Almera,Nissan,2026-09-04
4,Alice,Honda Civic,Honda,2026-09-05
5,Emma,Mazda 3,Mazda,2026-09-06
6,Frank,Ford Ranger,Ford,2026-09-07
7,Grace,Toyota Camry,Toyota,2026-09-08
8,Henry,Mazda CX-5,Mazda,2026-09-09
9,Jack,Toyota Yaris,Toyota,2026-09-10


Step 15 — Traversal หลายทอด

โจทย์:

คนที่ชอบรถเหมือน Alice เคยทดลองขับรถอะไรบ้าง?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name:'Alice'})
        -[:LIKES]->
        (:Car)
        <-[:LIKES]-
        (other:User)
        -[:TEST_DROVE]->
        (car:Car)

    WHERE other <> me

    RETURN DISTINCT
        me.name AS me,
        other.name AS other,
        car.name AS car

    ORDER BY other, car
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["me"],
        "-> คนที่ชอบเหมือนกัน:",
        record["other"],
        "-> ลองขับ:",
        record["car"]
    )

Alice -> คนที่ชอบเหมือนกัน: Bob -> ลองขับ: Honda Civic
Alice -> คนที่ชอบเหมือนกัน: Charlie -> ลองขับ: Mazda 3
Alice -> คนที่ชอบเหมือนกัน: David -> ลองขับ: Nissan Almera
Alice -> คนที่ชอบเหมือนกัน: Jack -> ลองขับ: Toyota Yaris


Step 16 — Aggregation

โจทย์:

User แต่ละคนทดลองขับรถกี่คัน?

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (u:User)
        -[:TEST_DROVE]->
        (c:Car)

    RETURN
        u.name AS user,
        count(c) AS total_cars

    ORDER BY total_cars DESC, user
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["user"],
        "=",
        record["total_cars"],
        "คัน"
    )

Alice = 2 คัน
Bob = 1 คัน
Charlie = 1 คัน
David = 1 คัน
Emma = 1 คัน
Frank = 1 คัน
Grace = 1 คัน
Henry = 1 คัน
Jack = 1 คัน


Step 17 — รถรุ่นไหนได้รับความนิยมมากที่สุด

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (:User)
        -[:LIKES]->
        (c:Car)

    RETURN
        c.name AS car,
        count(*) AS like_count

    ORDER BY like_count DESC, car
    """,
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

,car,like_count
0,Toyota Yaris,5
1,Toyota Corolla,4
2,Honda Accord,3
3,Honda City,3
4,Honda Civic,3
5,Mazda 3,3
6,Mazda CX-5,3
7,Nissan Almera,3
8,Toyota Camry,3
9,Ford Ranger,2


Step 18 — Mini Project: Recommendation System

โจทย์สำคัญที่สุด

แนะนำรถให้ Alice จากรถที่คนซึ่งชอบรถเหมือน Alice เคยลองขับ แต่ Alice ยังไม่เคยลองขับ

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (me:User {name:$name})
        -[:LIKES]->
        (:Car)
        <-[:LIKES]-
        (other:User)
        -[:TEST_DROVE]->
        (car:Car)

    WHERE other <> me

    AND NOT EXISTS {

        MATCH
            (me)-[:TEST_DROVE]->(car)

    }

    RETURN
        car.name AS recommendation,
        count(*) AS score,
        collect(DISTINCT other.name) AS test_driven_by

    ORDER BY
        score DESC,
        recommendation
    """,

    name="Alice",
    database_=DATABASE
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df

,recommendation,score,test_driven_by
0,Mazda 3,2,[Charlie]
1,Nissan Almera,1,[David]


ทำไม Mazda 3 ได้คะแนนสูงกว่า?


```
เพราะ

          Toyota Yaris
         /            \
Alice ─ LIKES        LIKES ─ Charlie
         \            /
          Toyota Corolla

Alice กับ Charlie ชอบรถตรงกัน 2 รุ่น

และ

Charlie ─── TEST_DROVE ───→ Mazda 3


ส่วน

Alice ─ LIKES → Toyota Corolla ← LIKES ─ David

Alice กับ David ชอบรถตรงกันแค่ 1 รุ่น

David ─── TEST_DROVE ───→ Nissan Almera


ดังนั้น

มี 2 เส้นทางจาก Alice ไปถึง Mazda 3        → score = 2
มี 1 เส้นทางจาก Alice ไปถึง Nissan Almera  → score = 1
```

Step 19 — ทำเป็น Python Function

เปลี่ยน Mini Project ให้ดูเหมือนโปรแกรมจริง

In [ ]:
def recommend_cars(name):

    query = """
    MATCH
        (me:User {name:$name})
        -[:LIKES]->
        (:Car)
        <-[:LIKES]-
        (other:User)
        -[:TEST_DROVE]->
        (car:Car)

    WHERE other <> me

    AND NOT EXISTS {

        MATCH
            (me)-[:TEST_DROVE]->(car)

    }

    RETURN
        car.name AS recommendation,
        count(*) AS score,
        collect(DISTINCT other.name) AS test_driven_by

    ORDER BY
        score DESC,
        recommendation
    """

    result = driver.execute_query(
        query,
        name=name,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records],
        columns=["recommendation", "score", "test_driven_by"]
    )

In [ ]:
recommend_cars("Alice")

,recommendation,score,test_driven_by
0,Mazda 3,2,[Charlie]
1,Nissan Almera,1,[David]


Step 20 — Function ค้นหานักเรียน

In [ ]:
def find_user(name):

    result = driver.execute_query(
        """
        MATCH (u:User {name:$name})

        RETURN
            u.name AS name,
            COUNT { (u)-[:LIKES]->() } AS liked_cars,
            COUNT { (u)-[:TEST_DROVE]->() } AS test_drives
        """,

        name=name,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบ User"

    return result.records[0].data()

In [ ]:
find_user("Bob")

{'name': 'Bob', 'liked_cars': 4, 'test_drives': 1}

Step 21 — Function ดูรถที่ User เคยทดลองขับ

In [ ]:
def test_driven_cars(name):

    result = driver.execute_query(
        """
        MATCH
            (u:User {name:$name})
            -[r:TEST_DROVE]->
            (c:Car)

        RETURN
            c.name AS car,
            c.brand AS brand,
            r.test_date AS test_date

        ORDER BY test_date
        """,

        name=name,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records],
        columns=["car", "brand", "test_date"]
    )

In [ ]:
test_driven_cars("Alice")

,car,brand,test_date
0,Toyota Yaris,Toyota,2026-09-01
1,Honda Civic,Honda,2026-09-05


Step 22 — ดู Graph ใน Neo4j Aura

หลังจากรันข้อมูลจาก Colab แล้ว ให้นักเรียนกลับไปที่ Neo4j Aura → Query

รัน


In [ ]:
#Step 23 — ตรวจจำนวน Node และ Relationship
result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

{'nodes': 20, 'relationships': 42}


In [ ]:
#Step 24 — ปิด Connection เมื่อเรียนเสร็จ

driver.close()

print("Neo4j connection closed")